# Proyecto 2: Machine Learning Pipeline con LisPy + scikit-learn

**Objetivo**: Pipeline completo de machine learning: carga de datos, preprocesamiento, entrenamiento de modelos, evaluación y visualización de resultados.

**Features demostradas**:
- `py:run` para definir y entrenar modelos de scikit-learn
- `numpy-fast.lp` para operaciones vectorizadas
- `py:->lisp` para traer métricas a Lisp
- `defpyfun` para crear funciones Python con parámetros capturados desde Lisp
- `persist.lp` para checkpoint de modelos entrenados
- Visualización con matplotlib

**Dataset**: Iris (scikit-learn built-in) y uno sintético para regresión.

## 1. Inicialización

In [ ]:
;; Cargar librerías necesarias
(load "lib/numpy-fast.lp")
(load "lib/persist.lp")
(py:import "sklearn")
(py:import "matplotlib.pyplot")
(py:import "numpy")
(py:import "pandas")

## 2. Clasificación — Iris Dataset

Pipeline completo de clasificación con Random Forest y comparación de modelos.

In [ ]:
(py:run "
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np
import pandas as pd

# Cargar datos
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names.tolist()

print(f'Dataset Iris: {X.shape[0]} muestras, {X.shape[1]} features')
print(f'Clases: {target_names}')
print(f'Features: {feature_names}')

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')
")

### 2.1 Comparación de Modelos desde Lisp

Definimos una función Lisp para evaluar cualquier clasificador y comparamos.

In [ ]:
;; Definir y evaluar modelos desde Lisp
(py:run "
modelos = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', random_state=42),
    'Regresión Logística': LogisticRegression(max_iter=200, random_state=42)
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    cv = cross_val_score(modelo, X, y, cv=5).mean()
    resultados[nombre] = {'accuracy': acc, 'cv_mean': cv}
    print(f'{nombre:25s}: Test={acc:.4f}, CV(5)={cv:.4f}')
")

### 2.2 Mejor Modelo — Reporte Detallado

Entrenamos el mejor modelo con más detalle.

In [ ]:
(py:run "
# Entrenar mejor modelo
rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

print('=== Reporte de Clasificación ===')
print(classification_report(y_test, y_pred, target_names=target_names))

print('=== Matriz de Confusión ===')
print(confusion_matrix(y_test, y_pred))

# Importancia de features
importancias = rf.feature_importances_
for i, (name, imp) in enumerate(zip(feature_names, importancias)):
    print(f'  {name:20s}: {imp:.4f}')
")

### 2.3 Visualización de Resultados

Matriz de confusión y importancia de features desde Lisp.

In [ ]:
(py:run "
import matplotlib.pyplot as plt
import seaborn as sns  # si está instalado

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Matriz de confusión
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, 
    display_labels=target_names, ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de Confusión - Random Forest')

# Importancia de features
axes[1].barh(feature_names, importancias, color='steelblue')
axes[1].set_title('Importancia de Features')
axes[1].set_xlabel('Importancia')

plt.tight_layout()
plt.savefig('ml-resultados.png', dpi=100)
plt.show()
")

## 3. Regresión — Datos Sintéticos

Creamos datos sintéticos para regresión y comparamos modelos.

In [ ]:
(py:run "
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

np.random.seed(42)

# Datos sintéticos: y = 3x^2 + 2x + 1 + ruido
X_reg = np.linspace(-5, 5, 200).reshape(-1, 1)
y_reg = 3 * X_reg.ravel()**2 + 2 * X_reg.ravel() + 1 + np.random.normal(0, 3, 200)

# Split
from sklearn.model_selection import train_test_split
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42)

print(f'Regresión: {len(Xr_train)} train, {len(Xr_test)} test')
")

In [ ]:
(py:run "
# Pipeline con features polinomiales
modelos_reg = {
    'Lineal (grado 1)': Pipeline([
        ('poly', PolynomialFeatures(degree=1)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Polinomial (grado 2)': Pipeline([
        ('poly', PolynomialFeatures(degree=2)),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Polinomial (grado 5)': Pipeline([
        ('poly', PolynomialFeatures(degree=5)),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0))
    ])
}

for nombre, pipe in modelos_reg.items():
    pipe.fit(Xr_train, yr_train)
    yp = pipe.predict(Xr_test)
    mse = mean_squared_error(yr_test, yp)
    r2 = r2_score(yr_test, yp)
    print(f'{nombre:25s}: MSE={mse:.4f}, R²={r2:.4f}')
")

In [ ]:
;; Visualizar regresiones
(py:run "
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

X_plot = np.linspace(-5, 5, 500).reshape(-1, 1)

for ax, (nombre, pipe) in zip(axes, modelos_reg.items()):
    y_plot = pipe.predict(X_plot)
    ax.scatter(Xr_train, yr_train, alpha=0.3, s=10, label='Train')
    ax.scatter(Xr_test, yr_test, alpha=0.5, s=10, label='Test')
    ax.plot(X_plot, y_plot, 'r-', linewidth=2, label='Predicción')
    ax.set_title(nombre)
    ax.set_xlabel('X')
    ax.set_ylabel('y')
    ax.legend()

plt.tight_layout()
plt.savefig('regresion-modelos.png', dpi=100)
plt.show()
")

## 4. defpyfun — Función Python con Captura Lisp

Creamos una función Python que captura parámetros definidos en Lisp.

In [ ]:
;; Definir parámetros de modelo en Lisp
(defvar *modelo-n-estimadores* 150)
(defvar *modelo-max-depth* 5)
(defvar *modelo-random-state* 42)

;; Crear función Python con parámetros capturados
(load "lib/defpyfun.lp")
(defpyfun 'crear-random-forest '(X y)
  :capture `((*modelo-n-estimadores* . "n_estimators")
             (*modelo-max-depth* . "max_depth")
             (*modelo-random-state* . "random_state")))

;; Usar la función desde Lisp
(py:run "
modelo_lisp = crear_random_forest(X_train, y_train)
y_pred_lisp = modelo_lisp.predict(X_test)
print(f'Modelo creado desde Lisp - Accuracy: {accuracy_score(y_test, y_pred_lisp):.4f}')
")

## 5. Checkpoint del Modelo Entrenado

In [ ]:
;; Guardar el mejor modelo con pickle
(py:run "
import pickle
with open('modelo-rf-iris.pkl', 'wb') as f:
    pickle.dump(rf, f)
print('Modelo guardado: modelo-rf-iris.pkl')
")

;; Demostrar carga
(py:run "
with open('modelo-rf-iris.pkl', 'rb') as f:
    rf_cargado = pickle.load(f)
print(f'Modelo cargado - Predicción de prueba: {rf_cargado.predict(X_test[:3])}')
print(f'Clases reales: {y_test[:3]}')
")

;; Guardar sesión completa
(py:save-session "checkpoint-ml.txt")
(format t "Sesión ML guardada~%")

## 6. Métricas a Lisp — Análisis Funcional

Extraemos métricas a Lisp para análisis funcional.

In [ ]:
;; Evaluar múltiples configuraciones desde Lisp
(defun evaluar-configuracion (n-est depth)
  (py:run (format nil "
from sklearn.ensemble import RandomForestClassifier
rf_cfg = RandomForestClassifier(n_estimators=~D, max_depth=~D, random_state=42)
rf_cfg.fit(X_train, y_train)
resultado_~D_~D = rf_cfg.score(X_test, y_test)
print(f'n=~D, depth=~D: {resultado_~D_~D:.4f}')" n-est depth n-est depth n-est depth n-est depth)))

;; Grid search manual desde Lisp
(format t "=== Grid Search desde Lisp ===~%")
(dolist (n-est '(50 100 150 200))
  (dolist (depth '(3 5 10 nil))
    (evaluar-configuracion n-est (or depth -1))))

## 7. Ejercicios Propuestos

1. Agregar KNN y Árbol de Decisión a la comparación de clasificadores
2. Implementar validación cruzada estratificada desde Lisp
3. Usar `defpyclass` para crear una clase `ModeloML` sincronizada Lisp/Python
4. Agregar GridSearchCV para optimización automática de hiperparámetros
5. Crear un ensemble (voting classifier) combinando los 3 modelos desde Lisp